<a href="https://colab.research.google.com/github/jacecard-stack/PY191_lab/blob/main/Week4Day2_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from IPython.display import HTML
HTML("<style>.container{width:100%!important;margin:auto}div.cell,div.input_area{padding:2px;margin:0}div.output_wrapper{padding:0;margin:0}</style>")

In [2]:
import copy

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, precision_recall_fscore_support

In [31]:
# --- 1. DATA PREPARATION ---
data = load_iris()

X_raw = torch.tensor(data.data, dtype=torch.float32)
print(X_raw.shape)
Y_raw = torch.tensor(data.target, dtype=torch.long)
print(np.unique(Y_raw))
X_train, X_test, Y_train, Y_test =  train_test_split(X_raw, Y_raw, test_size=0.2, random_state=42)
X_train, X_val, Y_train, Y_val =  train_test_split(X_train, Y_train, test_size=0.1, random_state=42)
print(X_train.shape, X_val.shape, X_test.shape)

scaler = StandardScaler()
X_train = torch.tensor(scaler.fit_transform(X_train), dtype=torch.float32) #You only normalize here
X_val = torch.tensor(scaler.transform(X_val), dtype=torch.float32)
X_test = torch.tensor(scaler.transform(X_test), dtype=torch.float32)

torch.Size([150, 4])
[0 1 2]
torch.Size([108, 4]) torch.Size([12, 4]) torch.Size([30, 4])


In [30]:
# --- 2. CREATE PYTORCH DATASETS & DATALOADERS ---
#merges the x and y of train val and test
train_dataset = TensorDataset(X_train, Y_train)
val_dataset = TensorDataset(X_val, Y_val)
test_dataset = TensorDataset(X_test, Y_test)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)


In [35]:
# --- 3. DEFINE THE MLP ---
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(SimpleMLP, self).__init__()
        self.hidden = nn.Linear(input_dim, hidden_dim)#layer 1 of the neural network
        self.output = nn.Linear(hidden_dim, output_dim)#output layer
        self.relu = nn.ReLU()


    def forward(self, x):
        x = self.relu(self.hidden(x))
        x = self.output(x)
        return x #x will be all the inputs we pass through the model through the hidden layer for an output


In [49]:
# --- 4. TRAINING SETUP ---
model = SimpleMLP(input_dim=X_train.shape[1], hidden_dim=4, output_dim=len(np.unique(Y_raw)))
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

print("training started")
MAX_EPOCHS = 100
for epoch in range(1, MAX_EPOCHS):
  model.train()
  train_loss = 0
  accuracy = 0
  for batch_X, batch_Y in train_loader:
      optimizer.zero_grad()
      pred = model(batch_X)
      loss = criterion(pred, batch_Y)
      loss.backward()
      optimizer.step()
      train_loss = train_loss + loss.item()*batch_X.size(0)
      train_pred = pred.argmax(dim=1)#argmax gives the index of the highest predicted probability
      #it will choose from the classes (in this case 3 classes)
      accuracy = accuracy + (train_pred == batch_Y).sum().item()
  avg_loss = train_loss / len(train_loader.dataset)
  avg_acc = accuracy / len(train_loader.dataset)

  model.eval()
  val_loss = 0
  val_acc = 0
  with torch.no_grad():
      for batch_X, batch_Y in val_loader:
          pred = model(batch_X)
          loss = criterion(pred, batch_Y)
          val_loss = val_loss + loss.item()*batch_X.size(0)
          val_pred = pred.argmax(dim=1)
          val_acc = val_acc + (val_pred == batch_Y).sum().item()
  avg_loss_val = val_loss / len(val_loader.dataset)
  avg_acc_val = val_acc / len(val_loader.dataset)*100
  print(f"Epoch {epoch} ---> TrainLoss: {avg_loss} ----> TrainAccuracy: {avg_acc}, ValLoss: {avg_loss_val}, ValAccuracy: {avg_acc_val}")


training started
Epoch 1 ---> TrainLoss: 1.2338942068594474 ----> TrainAccuracy: 0.3148148148148148, ValLoss: 1.301673412322998, ValAccuracy: 25.0
Epoch 2 ---> TrainLoss: 1.215151830955788 ----> TrainAccuracy: 0.3148148148148148, ValLoss: 1.2792123556137085, ValAccuracy: 25.0
Epoch 3 ---> TrainLoss: 1.1983645977797333 ----> TrainAccuracy: 0.3148148148148148, ValLoss: 1.258926510810852, ValAccuracy: 25.0
Epoch 4 ---> TrainLoss: 1.1833370173418964 ----> TrainAccuracy: 0.3148148148148148, ValLoss: 1.2409800291061401, ValAccuracy: 25.0
Epoch 5 ---> TrainLoss: 1.168734241414953 ----> TrainAccuracy: 0.3148148148148148, ValLoss: 1.2238603830337524, ValAccuracy: 25.0
Epoch 6 ---> TrainLoss: 1.1551011977372345 ----> TrainAccuracy: 0.3148148148148148, ValLoss: 1.2080435752868652, ValAccuracy: 25.0
Epoch 7 ---> TrainLoss: 1.142243879812735 ----> TrainAccuracy: 0.3148148148148148, ValLoss: 1.192743182182312, ValAccuracy: 25.0
Epoch 8 ---> TrainLoss: 1.129814346631368 ----> TrainAccuracy: 0.2962962

In [44]:
# --- 5. EVALUATION ---
model.eval()
test_acc = 0
all_pred = []
all_target = []
with torch.no_grad():
    for batch_X, batch_Y in val_loader:
      pred = model(batch_X)
      test_pred = pred.argmax(dim = 1)
      all_pred.extend(test_pred.cpu().numpy())
      all_target.extend(batch_Y.cpu().numpy())
target_name = data.target_names
print(classification_report(all_target, all_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.45      1.00      0.62         5
           2       1.00      0.25      0.40         4

    accuracy                           0.50        12
   macro avg       0.48      0.42      0.34        12
weighted avg       0.52      0.50      0.39        12



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# D. Early Stopping & Checkpointing Setup (Copy 4)

In [ ]:
# --- 5. EVALUATION --- (Copy 5)

In [ ]:
# --- Feature Importance via Input Gradient Analysis ---

In [3]:
# Train on some other standard datasets
from sklearn.datasets import load_wine, load_digits, load_breast_cancer
import numpy as np
# Copy 1, 2, 3, D, 5